# 🐝 Spelling Bee — Puzzle Analysis

Statistics derived from `data/nytbee_db.json` — the reference database of all past NYT Spelling Bee puzzles.
This notebook explores the structure and long-term trends of the game itself, independent of personal performance.

In [ ]:
import json
import string
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, date as date_type

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

C_FOUND   = "#F7C948"
C_MISSED  = "#E06B3C"
C_PANGRAM = "#9C27B0"
C_BLUE    = "#4A90D9"
C_BG      = "#FAFAFA"

ROOT      = Path("..")
NYTBEE_DB = ROOT / "data" / "nytbee_db.json"

with open(NYTBEE_DB) as f:
    nytbee = json.load(f)

rows = []
for date_str, v in nytbee.items():
    date     = datetime.strptime(date_str, "%Y-%m-%d")
    words    = [w.lower() for w in v.get("words",          [])]
    pangrams = [w.lower() for w in v.get("pangrams",       [])]
    letters  = [l.lower() for l in v.get("puzzle_letters", [])]
    center   = v.get("center_letter", "").lower()
    rows.append({
        "date":           date,
        "date_str":       date_str,
        "words":          words,
        "pangrams":       pangrams,
        "puzzle_letters": letters,
        "center_letter":  center,
        "n_words":        len(words),
        "n_pangrams":     len(pangrams),
        "year":           date.year,
        "month":          date.month,
    })

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
print(f"Loaded {len(df)} puzzles  ·  {df['date'].min().date()} → {df['date'].max().date()}")

## 1. Database Overview

In [ ]:
all_words       = [w for _, r in df.iterrows() for w in r["words"]]
unique_words    = len(set(all_words))
all_pangrams    = [p for _, r in df.iterrows() for p in r["pangrams"]]
unique_pangrams = len(set(all_pangrams))

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), facecolor=C_BG)
fig.suptitle("NYT Spelling Bee — Reference Database Overview", fontsize=13, fontweight="bold", y=1.05)

cards = [
    ("Puzzles in database",  f"{len(df):,}",                  C_BLUE),
    ("Unique words",         f"{unique_words:,}",              "#7B68EE"),
    ("Avg words per puzzle", f"{df['n_words'].mean():.1f}",    C_FOUND),
    ("Unique pangrams",      f"{unique_pangrams:,}",           C_PANGRAM),
]
for ax, (label, val, color) in zip(axes, cards):
    ax.set_facecolor("white")
    ax.text(0.5, 0.60, val,   ha="center", va="center", fontsize=26, fontweight="bold",
            color=color, transform=ax.transAxes)
    ax.text(0.5, 0.18, label, ha="center", va="center", fontsize=10.5, color="#555",
            transform=ax.transAxes)
    ax.axis("off")
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color("#E0E0E0"); sp.set_linewidth(1.5)

plt.tight_layout()
plt.show()

## 2. Puzzle Size Over Time

Word count per puzzle, with a 90-day rolling average. Are puzzles getting harder or easier?

In [ ]:
roll = df["n_words"].rolling(90, min_periods=15, center=True).mean()
yearly = df.groupby("year")["n_words"].mean()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.fill_between(df["date"], df["n_words"], alpha=0.12, color=C_BLUE)
ax.scatter(df["date"], df["n_words"], s=4, alpha=0.3, color=C_BLUE)
ax.plot(df["date"], roll, color=C_BLUE, linewidth=2.2, label="90-puzzle rolling avg")
ax.axhline(df["n_words"].mean(), color="#888", linestyle="--", linewidth=1,
           label=f"All-time avg  ({df['n_words'].mean():.1f} words)")

ax.set_ylabel("Words per puzzle")
ax.set_title("Puzzle Word Count Over Time")
ax.legend(frameon=False)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print("Yearly average word counts:")
for year, mean_val in yearly.items():
    bar = "█" * int(mean_val / 2)
    print(f"  {year}:  {mean_val:5.1f}  {bar}")

## 3. Pangram Count Distribution

Most puzzles have exactly one pangram. How often do multiple pangrams appear?

In [ ]:
pg_counts = df["n_pangrams"].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.bar(pg_counts.index.astype(str), pg_counts.values,
        color=C_PANGRAM, edgecolor="white")
ax1.set_xlabel("Pangrams in puzzle")
ax1.set_ylabel("Number of puzzles")
ax1.set_title("Pangram Count Distribution")
for i, (cnt, val) in enumerate(pg_counts.items()):
    ax1.text(i, val + 1, f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)

ax2.plot(pg_counts.index, pg_counts.cumsum() / len(df) * 100,
         marker="o", color=C_PANGRAM, linewidth=2, markersize=6)
ax2.set_xlabel("Pangrams in puzzle")
ax2.set_ylabel("Cumulative % of puzzles")
ax2.set_title("Cumulative Pangram Distribution")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax2.set_xticks(pg_counts.index)

plt.tight_layout()
plt.show()

most_pg = df.loc[df["n_pangrams"].idxmax()]
print(f"Most pangrams in one puzzle: {most_pg['n_pangrams']}  on {most_pg['date_str']}")
print(f"  Pangrams: {', '.join(most_pg['pangrams'])}")

## 4. Word Length Distribution

The minimum word length is 4. How long are Spelling Bee words in practice?
Pangrams, which use all 7 letters, tend to be longer.

In [ ]:
non_pangram_words = [
    w for _, r in df.iterrows()
    for w in r["words"] if w not in set(r["pangrams"])
]
all_pangrams_flat = [p for _, r in df.iterrows() for p in r["pangrams"]]

min_l   = min(len(w) for w in non_pangram_words + all_pangrams_flat)
max_l   = max(len(w) for w in non_pangram_words + all_pangrams_flat)
lengths = list(range(min_l, max_l + 1))

def count_by_len(words_list):
    c = Counter(len(w) for w in words_list)
    return [c.get(l, 0) for l in lengths]

np_vals = count_by_len(non_pangram_words)
pg_vals = count_by_len(all_pangrams_flat)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(lengths, np_vals, color=C_FOUND,   label="Regular words", edgecolor="white")
ax.bar(lengths, pg_vals, bottom=np_vals,  color=C_PANGRAM, label="Pangrams",  edgecolor="white")
ax.set_xlabel("Word length")
ax.set_ylabel("Total occurrences across all puzzles")
ax.set_title("Word Length Distribution Across All Puzzles")
ax.set_xticks(lengths)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

all_lens = [len(w) for w in non_pangram_words + all_pangrams_flat]
print(f"Word lengths — min: {min(all_lens)}, max: {max(all_lens)}, "
      f"mean: {np.mean(all_lens):.1f}, median: {int(np.median(all_lens))}")
print(f"Pangram lengths — mean: {np.mean([len(p) for p in all_pangrams_flat]):.1f}")

## 5. Letter Frequency

Which letters appear most as center letters (the mandatory letter every word must contain)?
And which letters appear most as outer letters?
Some letters may never appear at all — the NYT avoids exotic letter combinations.

In [ ]:
all_letters  = list(string.ascii_lowercase)
center_ctr   = Counter(df["center_letter"])
outer_ctr    = Counter(
    l for _, r in df.iterrows()
    for l in r["puzzle_letters"]
    if l != r["center_letter"]
)

center_vals = [center_ctr.get(l, 0) for l in all_letters]
outer_vals  = [outer_ctr.get(l, 0)  for l in all_letters]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

colors_c = ["#DDD" if v == 0 else C_MISSED for v in center_vals]
ax1.bar(all_letters, center_vals, color=colors_c, edgecolor="white")
ax1.set_ylabel("Times used as center letter")
ax1.set_title("Center Letter Frequency")
for i, (l, v) in enumerate(zip(all_letters, center_vals)):
    if v == 0:
        ax1.text(i, 2, "✕", ha="center", va="bottom", fontsize=8, color="#BBB")

colors_o = ["#DDD" if v == 0 else C_BLUE for v in outer_vals]
ax2.bar(all_letters, outer_vals, color=colors_o, edgecolor="white")
ax2.set_ylabel("Times used as outer letter")
ax2.set_title("Outer Letter Frequency")

plt.tight_layout()
plt.show()

missing_center = [l.upper() for l in all_letters if center_ctr.get(l, 0) == 0]
missing_outer  = [l.upper() for l in all_letters if outer_ctr.get(l, 0)  == 0]
if missing_center:
    print(f"Never center: {', '.join(missing_center)}")
if missing_outer:
    print(f"Never outer:  {', '.join(missing_outer)}")

## 6. Puzzle Richness by Center Letter

Some center letters unlock more words than others. More words = richer puzzle.
Bars show the mean; error bars extend to the min and max.

In [ ]:
cl_stats = (
    df.groupby("center_letter")["n_words"]
    .agg(mean="mean", min="min", max="max", count="count")
    .query("count >= 5")
    .sort_values("mean", ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(cl_stats))
ax.bar(x, cl_stats["mean"], color=C_BLUE, edgecolor="white", label="Mean words")
ax.errorbar(
    x, cl_stats["mean"],
    yerr=[cl_stats["mean"] - cl_stats["min"], cl_stats["max"] - cl_stats["mean"]],
    fmt="none", ecolor="#888", capsize=3, elinewidth=1,
)
ax.axhline(df["n_words"].mean(), color="#888", linestyle="--", linewidth=1,
           label=f"Overall avg  ({df['n_words'].mean():.1f})")
ax.set_xticks(x)
ax.set_xticklabels(cl_stats.index.str.upper(), fontsize=10)
ax.set_ylabel("Words per puzzle")
ax.set_title("Puzzle Word Count by Center Letter  (mean ± range)")
ax.legend(frameon=False)

for i, (letter, row) in enumerate(cl_stats.iterrows()):
    ax.text(i, row["min"] - 1.5, f"n={int(row['count'])}", ha="center",
            fontsize=6.5, color="#888", va="top")

plt.tight_layout()
plt.show()

## 7. Most Recurring Words

Some words appear in multiple puzzles because they can be formed from different 7-letter sets.
The most common pangrams are especially interesting — they re-appear with different center letters.

In [ ]:
word_ctr    = Counter(w for _, r in df.iterrows() for w in r["words"])
pangram_ctr = Counter(p for _, r in df.iterrows() for p in r["pangrams"])

top_words   = word_ctr.most_common(30)
top_pangrams = pangram_ctr.most_common(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 9))

words, counts = zip(*top_words)
ax1.barh(range(len(words)), counts, color=C_FOUND, edgecolor="white")
ax1.set_yticks(range(len(words)))
ax1.set_yticklabels([w.upper() for w in words], fontsize=8.5, fontfamily="monospace")
ax1.invert_yaxis()
ax1.set_xlabel("Appearances across puzzles")
ax1.set_title("Top 30 Most Recurring Words")
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar, count in zip(ax1.patches, counts):
    ax1.text(count + 0.05, bar.get_y() + bar.get_height() / 2,
             f" {count}", va="center", fontsize=8)

pwords, pcounts = zip(*top_pangrams)
ax2.barh(range(len(pwords)), pcounts, color=C_PANGRAM, edgecolor="white")
ax2.set_yticks(range(len(pwords)))
ax2.set_yticklabels([w.upper() for w in pwords], fontsize=8.5, fontfamily="monospace")
ax2.invert_yaxis()
ax2.set_xlabel("Appearances as pangram")
ax2.set_title("Top 15 Most Recurring Pangrams")
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar, count in zip(ax2.patches, pcounts):
    ax2.text(count + 0.05, bar.get_y() + bar.get_height() / 2,
             f" {count}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 8. Distinct-Letter Count Distribution by Year

Each word uses between 4 and 7 distinct letters (7 = pangram territory).
Has the distribution shifted over the years — more complex words recently?

In [ ]:
by_year_n = defaultdict(lambda: defaultdict(int))
for _, row in df.iterrows():
    for w in row["words"]:
        n = len(set(w))
        if 2 <= n <= 7:
            by_year_n[row["year"]][n] += 1

years  = sorted(by_year_n)
ns     = list(range(2, 8))
cmap   = plt.colormaps["YlOrRd"]
colors = [cmap(0.15 + 0.13 * i) for i in range(len(ns))]

fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(len(years))
width = 0.11

for i, (n, color) in enumerate(zip(ns, colors)):
    vals = [
        by_year_n[y].get(n, 0) / sum(by_year_n[y].values()) * 100
        for y in years
    ]
    ax.bar(x + i * width - (len(ns) - 1) * width / 2, vals, width,
           label=f"{n} distinct", color=color, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("Share of words (%)")
ax.set_title("Word Distribution by Distinct-Letter Count, per Year")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.legend(frameon=False, title="Distinct letters", ncol=3, fontsize=9)
plt.tight_layout()
plt.show()